# 12. 판정요지 중복 정보 제거 및 사례 검색 비교

11번 실험의 후속 분석입니다. 판정요지 문장을 해설서 지식베이스와 비교해 중복 가능성이 높은 문장과 제품별 추가 판단 정보를 분리합니다.

비교 대상은 제품 속성만(product-only), 제품 속성+전체 판정요지(full ruling), 제품 속성+중복 제거 판정요지(residual ruling)입니다. 평가는 동일한 4-fold StratifiedGroupKFold에서 사례 검색의 MRR, Recall@5/10, nDCG@10으로 수행합니다. 이 지표는 사례 검색 품질이며 HS 코드 분류 정확도와 구분합니다.

중복 임계값은 0.72, 0.78, 0.84를 비교하며, 특정 임계값을 정답처럼 간주하지 않습니다.

In [ ]:
!pip -q install -U sentence-transformers
import re, hashlib
import numpy as np
import pandas as pd
from urllib.parse import quote
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedGroupKFold


## 1. 데이터 불러오기

In [ ]:
BASE = 'https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/'
cases_file = quote('완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv',safe='')
data_all = pd.read_csv(BASE+cases_file).fillna('')
kb = pd.read_json(BASE+'hs_knowledge_base.jsonl',lines=True).fillna('')
input_cols = ['Product Name','Main Ingredients (KR)','Main Ingredients (EN)','Product Form and Packaging','Intended Use']
target_col,summary_col = 'HS Code','Preprocessed Ruling Summary'
required = [target_col]+input_cols+[summary_col]
missing = [c for c in required if c not in data_all.columns]
if missing: raise ValueError(f'필수 컬럼 누락: {missing}')
data_all[required] = data_all[required].astype(str).replace('nan','').fillna('')
data_all[target_col] = data_all[target_col].str.strip()
data_all['query_text'] = data_all[input_cols].agg(' '.join,axis=1).str.strip()
data_all['duplicate_group'] = data_all[input_cols].astype(str).agg('||'.join,axis=1).map(lambda x:hashlib.md5(x.encode()).hexdigest())
eligible = data_all.groupby(target_col).filter(lambda x:len(x)>=4)
group_counts = eligible.groupby(target_col)['duplicate_group'].nunique()
labels = sorted(group_counts[group_counts>=4].index.astype(str))
data = eligible[eligible[target_col].isin(labels)].copy().reset_index(drop=True)
kb['document_text'] = kb['document_text'].astype(str)
print('사례:',len(data),'| HS 코드:',data[target_col].nunique(),'| KB 문서:',len(kb))
display(data[[target_col]+input_cols+[summary_col]].head(3))

## 2. 판정요지 문장 단위 중복 탐지

각 판정요지를 문장 단위로 나눈 뒤 SBERT 임베딩을 생성합니다. 각 문장과 해설서 문서의 최대 코사인 유사도를 계산하고, 임계값 이상인 문장을 중복 가능 문장으로 표시합니다. 의미가 비슷하더라도 실제로 같은 법적 판단인지 자동 확정할 수 없으므로 결과는 검토용으로 해석합니다.

In [ ]:
MODEL_NAME = 'intfloat/multilingual-e5-base'
model = SentenceTransformer(MODEL_NAME)
kb_texts = kb['document_text'].astype(str).tolist()
kb_emb = model.encode(['passage: '+x for x in kb_texts],batch_size=32,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
def split_sentences(text):
    text = re.sub(r'\s+',' ',str(text)).strip()
    parts = re.split(r'(?<=[.!?。])\s+|(?<=다\.)\s*',text)
    parts = [x.strip(' ,;:') for x in parts if len(x.strip())>=12]
    if len(parts)<=1 and len(text)>350: parts = [text[i:i+250] for i in range(0,len(text),220)]
    return parts if parts else ([text] if text else [])
sentence_rows = []
for row_id,row in data.iterrows():
    for sent_id,sent in enumerate(split_sentences(row[summary_col])):
        sentence_rows.append({'row_index':row_id,'sentence_id':sent_id,'sentence':sent})
sentences = pd.DataFrame(sentence_rows)
sent_emb = model.encode(['query: '+x for x in sentences['sentence'].tolist()],batch_size=32,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
sim_matrix = sent_emb@kb_emb.T
best_idx = sim_matrix.argmax(axis=1)
sentences['max_kb_similarity'] = sim_matrix.max(axis=1)
sentences['matched_knowledge_id'] = kb.iloc[best_idx]['knowledge_id'].to_numpy()
sentences['matched_source_sheet'] = kb.iloc[best_idx]['source_sheet'].to_numpy()
sentences['matched_rule_type'] = kb.iloc[best_idx]['rule_type'].to_numpy()
sentences['matched_document'] = kb.iloc[best_idx]['document_text'].to_numpy()
THRESHOLDS = [0.72,0.78,0.84]
for threshold in THRESHOLDS: sentences[f'redundant_{threshold}'] = sentences['max_kb_similarity']>=threshold
display(sentences.sort_values('max_kb_similarity',ascending=False).head(20))

## 3. 중복 비율 및 잔여 판정요지 생성

In [ ]:
summary_rows = []
for threshold in THRESHOLDS:
    flag = f'redundant_{threshold}'
    summary_rows.append({'threshold':threshold,'sentence_n':len(sentences),'redundant_sentence_n':int(sentences[flag].sum()),'redundant_sentence_ratio':float(sentences[flag].mean()) if len(sentences) else 0})
redundancy_metrics = pd.DataFrame(summary_rows)
for threshold in THRESHOLDS:
    residual = sentences.loc[~sentences[f'redundant_{threshold}']].groupby('row_index')['sentence'].apply(' '.join)
    data[f'residual_ruling_{threshold}'] = data.index.map(residual).fillna('')
    data[f'full_case_{threshold}'] = (data['query_text']+' '+data[summary_col]).str.strip()
    data[f'residual_case_{threshold}'] = (data['query_text']+' '+data[f'residual_ruling_{threshold}']).str.strip()
display(redundancy_metrics)
display(sentences.groupby('matched_source_sheet')['max_kb_similarity'].agg(['count','mean','max']).sort_values('mean',ascending=False).round(3))
display(data[[target_col,summary_col,'residual_ruling_0.78']].head(10))

## 4. Fold별 사례 검색 평가

평가 제품의 판정요지와 HS 코드는 검색 질의에 사용하지 않습니다. 검색 문서에는 각 Fold의 학습 사례만 포함합니다. 정답 사례는 동일 HS 코드의 학습 사례로 정의합니다.

In [ ]:
N_SPLITS = 4
cv = StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=42)
def retrieval_metrics(sim, relevant, ks=(5,10)):
    order = np.argsort(sim)[::-1]
    rel = np.asarray(relevant,dtype=int)[order]
    hits = np.where(rel==1)[0]
    mrr = 1/(hits[0]+1) if len(hits) else 0.0
    out = {'mrr':mrr}
    for k in ks: out[f'recall@{k}'] = float(rel[:k].sum()/max(rel.sum(),1))
    k = 10
    gains = rel[:k]/np.log2(np.arange(2,k+2))
    ideal = np.sort(rel)[::-1][:k]/np.log2(np.arange(2,k+2))
    out['ndcg@10'] = float(gains.sum()/ideal.sum()) if ideal.sum()>0 else 0.0
    return out
def evaluate_variant(train,test,doc_col,model):
    docs = train[doc_col].astype(str).tolist()
    queries = test['query_text'].astype(str).tolist()
    doc_emb = model.encode(['passage: '+x for x in docs],batch_size=32,show_progress_bar=False,normalize_embeddings=True,convert_to_numpy=True)
    query_emb = model.encode(['query: '+x for x in queries],batch_size=32,show_progress_bar=False,normalize_embeddings=True,convert_to_numpy=True)
    rows = []
    train_labels = train[target_col].astype(str).to_numpy()
    for i,row in enumerate(test.itertuples()):
        relevant = train_labels==str(getattr(row,target_col.replace(' ','_')))
        rows.append(retrieval_metrics(doc_emb@query_emb[i],relevant))
    return rows
variant_cols = {'product_only':'query_text','full_ruling':'full_case_0.78','residual_ruling':'residual_case_0.78'}
eval_rows,fold_rows = [],[]
for fold,(tr,te) in enumerate(cv.split(data['query_text'],data[target_col],groups=data['duplicate_group']),1):
    train,test = data.iloc[tr].copy(),data.iloc[te].copy()
    overlap = len(set(train['duplicate_group'])&set(test['duplicate_group']))
    fold_rows.append({'fold':fold,'train_n':len(train),'test_n':len(test),'group_overlap':overlap})
    for variant,doc_col in variant_cols.items():
        vals = evaluate_variant(train,test,doc_col,model)
        for value in vals: eval_rows.append({'fold':fold,'variant':variant,**value})
fold_audit = pd.DataFrame(fold_rows)
retrieval_results = pd.DataFrame(eval_rows)
retrieval_summary = retrieval_results.groupby('variant')[['mrr','recall@5','recall@10','ndcg@10']].mean().reset_index()
display(fold_audit)
display(retrieval_summary.sort_values('mrr',ascending=False))

## 5. 임계값 민감도 분석

0.78 외 임계값에서도 잔여 판정요지의 검색 성능을 확인합니다. 같은 평가 Fold에서 여러 임계값을 비교하므로 결과는 탐색적 분석으로 보고, 최종 임계값 선택에는 별도 검증 데이터가 필요합니다.

In [ ]:
threshold_rows = []
for threshold in THRESHOLDS:
    col = f'residual_case_{threshold}'
    for fold,(tr,te) in enumerate(cv.split(data['query_text'],data[target_col],groups=data['duplicate_group']),1):
        vals = evaluate_variant(data.iloc[tr],data.iloc[te],col,model)
        for value in vals: threshold_rows.append({'threshold':threshold,'fold':fold,**value})
threshold_results = pd.DataFrame(threshold_rows)
threshold_summary = threshold_results.groupby('threshold')[['mrr','recall@5','recall@10','ndcg@10']].mean().reset_index()
display(threshold_summary)

## 6. 결과 저장 및 해석 유의사항

- `12_sentence_overlap.csv`: 판정요지 문장별 해설서 최대 유사도 및 매칭 문서
- `12_redundancy_metrics.csv`: 임계값별 중복 문장 비율
- `12_retrieval_metrics.csv`: 제품 속성만/전체 판정요지/중복 제거 판정요지 비교
- `12_threshold_sensitivity.csv`: 임계값별 검색 성능
- `12_fold_audit.csv`: Fold 간 중복 그룹 누수 점검

중복 문장 판정은 SBERT 유사도에 기반한 자동 탐지이며 법적·의미적 동일성을 확정하지 않습니다. 검색 성능은 사례 검색의 품질이며 HS 코드 분류 성능과 동일하지 않습니다.

In [ ]:
sentences.to_csv('12_sentence_overlap.csv',index=False,encoding='utf-8-sig')
redundancy_metrics.to_csv('12_redundancy_metrics.csv',index=False,encoding='utf-8-sig')
retrieval_results.to_csv('12_retrieval_metrics.csv',index=False,encoding='utf-8-sig')
threshold_results.to_csv('12_threshold_sensitivity.csv',index=False,encoding='utf-8-sig')
fold_audit.to_csv('12_fold_audit.csv',index=False,encoding='utf-8-sig')
print('12번 실험 결과 저장 완료')